# 🤖 Clase 2: Sistemas Multi-Agente

## Bienvenido a la Semana 3, Clase 2

En esta clase aprenderás:
- ✅ Arquitecturas multi-agente
- ✅ Patrones de coordinación
- ✅ Agentes especializados
- ✅ Comunicación entre agentes
- ✅ Casos de uso reales
- ✅ Implementación práctica

---

In [ ]:
!pip install langgraph langchain-openai -q

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List
import operator

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model="gpt-4", temperature=0.7)
print("✅ Configuración completada")

## 🏗️ Parte 1: Arquitecturas Multi-Agente

### 1. Arquitectura Jerárquica

```
    Coordinador
    /    |    \
  A1    A2    A3
```

**Uso**: Cuando hay un agente líder que coordina a otros.

### 2. Arquitectura Pipeline

```
A1 → A2 → A3 → Resultado
```

**Uso**: Procesamiento secuencial especializado.

### 3. Arquitectura Colaborativa

```
   A1 ←→ A2
    ↕     ↕
   A3 ←→ A4
```

**Uso**: Agentes que trabajan juntos compartiendo información.

## 🔄 Parte 2: Sistema Pipeline - Análisis de Contenido

In [ ]:
# Estado compartido
class ContentState(TypedDict):
    original_text: str
    summary: str
    keywords: List[str]
    sentiment: str
    final_report: str

# Agente 1: Resumidor
def summarizer_agent(state: ContentState) -> ContentState:
    """Resume el texto."""
    print("📝 Agente Resumidor trabajando...")
    
    prompt = ChatPromptTemplate.from_template(
        "Resume este texto en 2-3 oraciones:\n{text}"
    )
    chain = prompt | llm | StrOutputParser()
    state["summary"] = chain.invoke({"text": state["original_text"]})
    
    return state

# Agente 2: Extractor de Keywords
def keyword_agent(state: ContentState) -> ContentState:
    """Extrae palabras clave."""
    print("🔑 Agente de Keywords trabajando...")
    
    prompt = ChatPromptTemplate.from_template(
        "Extrae 5 palabras clave de este texto (separadas por comas):\n{text}"
    )
    chain = prompt | llm | StrOutputParser()
    keywords_str = chain.invoke({"text": state["original_text"]})
    state["keywords"] = [k.strip() for k in keywords_str.split(",")]
    
    return state

# Agente 3: Analizador de Sentimiento
def sentiment_agent(state: ContentState) -> ContentState:
    """Analiza el sentimiento."""
    print("😊 Agente de Sentimiento trabajando...")
    
    prompt = ChatPromptTemplate.from_template(
        "Analiza el sentimiento de este texto (positivo/negativo/neutral):\n{text}"
    )
    chain = prompt | llm | StrOutputParser()
    state["sentiment"] = chain.invoke({"text": state["original_text"]})
    
    return state

# Agente 4: Generador de Reporte
def report_agent(state: ContentState) -> ContentState:
    """Genera reporte final."""
    print("📊 Agente de Reporte generando...")
    
    state["final_report"] = f"""# Análisis de Contenido

## Resumen
{state['summary']}

## Palabras Clave
{', '.join(state['keywords'])}

## Sentimiento
{state['sentiment']}
"""
    return state

print("✅ Agentes del pipeline definidos")

In [ ]:
# Crear grafo pipeline
pipeline_workflow = StateGraph(ContentState)

# Agregar agentes
pipeline_workflow.add_node("summarizer", summarizer_agent)
pipeline_workflow.add_node("keywords", keyword_agent)
pipeline_workflow.add_node("sentiment", sentiment_agent)
pipeline_workflow.add_node("report", report_agent)

# Flujo secuencial
pipeline_workflow.set_entry_point("summarizer")
pipeline_workflow.add_edge("summarizer", "keywords")
pipeline_workflow.add_edge("keywords", "sentiment")
pipeline_workflow.add_edge("sentiment", "report")
pipeline_workflow.add_edge("report", END)

pipeline_app = pipeline_workflow.compile()

print("✅ Pipeline multi-agente creado")

In [ ]:
# Probar el pipeline
texto_ejemplo = """La inteligencia artificial está transformando la educación de manera positiva. 
Los estudiantes ahora tienen acceso a tutores personalizados impulsados por IA que se adaptan a su 
ritmo de aprendizaje. Las herramientas de IA también ayudan a los profesores a identificar áreas 
donde los estudiantes necesitan más apoyo. Sin embargo, es importante mantener el equilibrio entre 
la tecnología y la interacción humana en el aula."""

print("🚀 Ejecutando pipeline multi-agente...\n")
print("="*80)

result = pipeline_app.invoke({
    "original_text": texto_ejemplo,
    "summary": "",
    "keywords": [],
    "sentiment": "",
    "final_report": ""
})

print("\n📄 REPORTE FINAL:")
print("="*80)
print(result["final_report"])

## 👥 Parte 3: Sistema Jerárquico - Equipo de Investigación

In [ ]:
# Estado para sistema jerárquico
class ResearchTeamState(TypedDict):
    task: str
    researcher_output: str
    analyst_output: str
    writer_output: str
    coordinator_decision: str
    final_output: str

# Agente Investigador
def researcher(state: ResearchTeamState) -> ResearchTeamState:
    """Investiga el tema."""
    print("🔍 Investigador: Buscando información...")
    
    prompt = ChatPromptTemplate.from_template(
        "Como investigador experto, investiga sobre: {task}. Proporciona datos y hechos clave."
    )
    chain = prompt | llm | StrOutputParser()
    state["researcher_output"] = chain.invoke({"task": state["task"]})
    
    return state

# Agente Analista
def analyst(state: ResearchTeamState) -> ResearchTeamState:
    """Analiza la información."""
    print("📊 Analista: Analizando datos...")
    
    prompt = ChatPromptTemplate.from_template(
        "Como analista, analiza esta información y extrae insights:\n{info}"
    )
    chain = prompt | llm | StrOutputParser()
    state["analyst_output"] = chain.invoke({"info": state["researcher_output"]})
    
    return state

# Agente Escritor
def writer(state: ResearchTeamState) -> ResearchTeamState:
    """Escribe el contenido."""
    print("✍️ Escritor: Redactando...")
    
    prompt = ChatPromptTemplate.from_template(
        "Como escritor profesional, crea un artículo basado en:\n{analysis}"
    )
    chain = prompt | llm | StrOutputParser()
    state["writer_output"] = chain.invoke({"analysis": state["analyst_output"]})
    
    return state

# Agente Coordinador
def coordinator(state: ResearchTeamState) -> ResearchTeamState:
    """Coordina y decide."""
    print("👔 Coordinador: Revisando trabajo del equipo...")
    
    # El coordinador revisa y decide si está listo
    state["coordinator_decision"] = "approved"
    state["final_output"] = f"""# {state['task']}

{state['writer_output']}

---
*Aprobado por el Coordinador*
"""
    
    return state

print("✅ Equipo de investigación definido")

In [ ]:
# Crear grafo jerárquico
team_workflow = StateGraph(ResearchTeamState)

team_workflow.add_node("researcher", researcher)
team_workflow.add_node("analyst", analyst)
team_workflow.add_node("writer", writer)
team_workflow.add_node("coordinator", coordinator)

# Flujo: Investigador → Analista → Escritor → Coordinador
team_workflow.set_entry_point("researcher")
team_workflow.add_edge("researcher", "analyst")
team_workflow.add_edge("analyst", "writer")
team_workflow.add_edge("writer", "coordinator")
team_workflow.add_edge("coordinator", END)

team_app = team_workflow.compile()

print("✅ Sistema jerárquico creado")

In [ ]:
# Ejecutar equipo
task = "El impacto de la IA en el mercado laboral"

print(f"🚀 Equipo trabajando en: {task}\n")
print("="*80)

result = team_app.invoke({
    "task": task,
    "researcher_output": "",
    "analyst_output": "",
    "writer_output": "",
    "coordinator_decision": "",
    "final_output": ""
})

print("\n📄 RESULTADO FINAL:")
print("="*80)
print(result["final_output"])

## 🔄 Parte 4: Sistema Colaborativo - Debate de Agentes

In [ ]:
# Sistema donde agentes debaten un tema
class DebateState(TypedDict):
    topic: str
    arguments_for: Annotated[List[str], operator.add]
    arguments_against: Annotated[List[str], operator.add]
    round: int
    max_rounds: int
    conclusion: str

def advocate_agent(state: DebateState) -> DebateState:
    """Agente a favor."""
    print(f"👍 Defensor (Ronda {state['round']})...")
    
    prompt = ChatPromptTemplate.from_template(
        "Da un argumento A FAVOR de: {topic}"
    )
    chain = prompt | llm | StrOutputParser()
    argument = chain.invoke({"topic": state["topic"]})
    state["arguments_for"].append(argument)
    
    return state

def critic_agent(state: DebateState) -> DebateState:
    """Agente en contra."""
    print(f"👎 Crítico (Ronda {state['round']})...")
    
    prompt = ChatPromptTemplate.from_template(
        "Da un argumento EN CONTRA de: {topic}"
    )
    chain = prompt | llm | StrOutputParser()
    argument = chain.invoke({"topic": state["topic"]})
    state["arguments_against"].append(argument)
    
    state["round"] += 1
    return state

def judge_agent(state: DebateState) -> DebateState:
    """Juez que concluye."""
    print("⚖️ Juez: Evaluando argumentos...")
    
    prompt = ChatPromptTemplate.from_template(
        """Evalúa este debate sobre: {topic}

Argumentos a favor:
{args_for}

Argumentos en contra:
{args_against}

Da una conclusión balanceada."""
    )
    chain = prompt | llm | StrOutputParser()
    state["conclusion"] = chain.invoke({
        "topic": state["topic"],
        "args_for": "\n".join(state["arguments_for"]),
        "args_against": "\n".join(state["arguments_against"])
    })
    
    return state

def should_continue_debate(state: DebateState) -> str:
    """Decide si continuar el debate."""
    if state["round"] < state["max_rounds"]:
        return "continue"
    return "judge"

print("✅ Sistema de debate definido")

In [ ]:
# Crear grafo de debate
debate_workflow = StateGraph(DebateState)

debate_workflow.add_node("advocate", advocate_agent)
debate_workflow.add_node("critic", critic_agent)
debate_workflow.add_node("judge", judge_agent)

debate_workflow.set_entry_point("advocate")
debate_workflow.add_edge("advocate", "critic")

# Decisión: continuar debate o ir al juez
debate_workflow.add_conditional_edges(
    "critic",
    should_continue_debate,
    {
        "continue": "advocate",
        "judge": "judge"
    }
)

debate_workflow.add_edge("judge", END)

debate_app = debate_workflow.compile()

print("✅ Sistema de debate creado")

In [ ]:
# Ejecutar debate
topic = "El trabajo remoto debería ser permanente"

print(f"⚖️ Debate: {topic}\n")
print("="*80)

result = debate_app.invoke({
    "topic": topic,
    "arguments_for": [],
    "arguments_against": [],
    "round": 1,
    "max_rounds": 2,
    "conclusion": ""
})

print("\n📊 CONCLUSIÓN DEL JUEZ:")
print("="*80)
print(result["conclusion"])

## 💡 Ejercicios Prácticos

In [ ]:
# Ejercicio 1: Crea un sistema multi-agente para atención al cliente
# Agentes:
# 1. Clasificador: Clasifica el tipo de consulta
# 2. Soporte Técnico: Maneja problemas técnicos
# 3. Ventas: Maneja consultas de ventas
# 4. General: Maneja otras consultas

# 👉 Tu código aquí
class CustomerServiceState(TypedDict):
    query: str
    category: str
    response: str

# Implementa los agentes...
print("Ejercicio 1: Sistema de atención al cliente")

## 🎓 Resumen

### Arquitecturas Multi-Agente

1. **Jerárquica**: Coordinador + especialistas
2. **Pipeline**: Procesamiento secuencial
3. **Colaborativa**: Agentes que interactúan

### Cuándo Usar Multi-Agente

✅ **Usar cuando**:
- Tareas complejas divisibles
- Necesitas especialización
- Quieres paralelizar trabajo
- Requieres diferentes perspectivas

❌ **No usar cuando**:
- Tarea simple
- Un solo agente es suficiente
- Overhead no justificado

### Mejores Prácticas

- 🎯 Define roles claros
- 📊 Usa estado compartido
- 🔄 Implementa coordinación
- 🐛 Debuggea con LangSmith
- ⏱️ Limita iteraciones

### Próxima Semana

En **Semana 4** aprenderemos:
- 🧪 Testing para LLMs
- 📊 Métricas de evaluación
- 🚀 Deployment
- 🎉 Demo Day

---

**¡Excelente trabajo! 🚀**